In [ ]:
import pandas as pd
import requests
from urllib.parse import urlencode,urlunsplit

In [ ]:
SCHEME = 'https'
BASE = 'api.worldbank.org/v2/'

In [ ]:
paises = []
indicator = 'SE.ADT.LITR.ZS'
start_year = '2013'
end_year = '2023'
lista = []
for country in ['all']:
    path= f'/country/{country}/indicator/{indicator}'
    query_dict = {'date':f'{start_year}:{end_year}', 'format':'json', 'per_page':10000}
    if query_dict is None:
        query_dict = {}
    query = urlencode(query_dict,safe=':')
    url=  urlunsplit((SCHEME,BASE,path,query,''))
    response = requests.get(url)
    a = response.json()[1]
a

In [ ]:
def fetch_indicator(indicator: str,
                    countries: list,start_year:int,
                    end_year: int) -> pd.DataFrame:
    paises = []
    for country in countries:
        path= f'/country/{country}/indicator/{indicator}'
        if end_year is None:
            info = f'{start_year}'
            fin = 1
        else:
            info = f'{start_year}:{end_year}'
            fin = int(end_year)-int(start_year)+1
        query_dict = {'date':f'{info}', 'format':'json', 'per_page':10000}
        if query_dict is None:
            query_dict = {}
        query = urlencode(query_dict,safe=':')
        url=  urlunsplit((SCHEME,BASE,path,query,''))
        response = requests.get(url)
        res_json = response.json()[1]
        if response.status_code != 404:
            for i in range(0,len(res_json)):
                indicador = res_json[i]['indicator']['id']
                value = res_json[i]['value']
                pais = res_json[i]['countryiso3code']
                year = res_json[i]['date']
                paises.append({'indicator':indicador,'country':pais,'value':value,'year':year})

    return pd.DataFrame(paises)
df = fetch_indicator('SE.ADT.LITR.ZS',["ESP", "FRA", "DEU", "USA", "BRA", "CHN", "ZAF", "NGA", "IND", "KEN"],'2013','2023')
df

In [ ]:
df2 = df.groupby('country')['value'].mean().reset_index().sort_values('value',ascending=False)
df2.rename(columns={'value':'iteracy_mean'})

In [ ]:
df_23 = df.loc[df['year']=='2023',['country','value']]
df_13 = df.loc[df['year']=='2013',['country','value']]
df_ambos = df_23.merge(df_13, on='country')
df_ambos['diferencia']= abs(df_ambos['value_x'] -df_ambos['value_y'])
df_ambos.dropna()

In [ ]:
indicadores=['SE.ADT.LITR.ZS','NY.GDP.PCAP.CD']
countries = ['all']
dfs = []
for indicador in indicadores:
    i=fetch_indicator(indicador,countries,'2021',None)
    dfs.append(i)
iteracy_2021= dfs[0].rename(columns={'value':'value_iteracy'})
iteracy_2021

In [ ]:
gpd_pc_2021 = dfs[1].rename(columns={'value':'value_gpd'})
gpd_pc_2021

In [ ]:
df= iteracy_2021.merge(gpd_pc_2021,on=['country','year'])
df

In [ ]:
df['value_iteracy'].corr(df['value_gpd'])

In [ ]:
students = pd.read_csv('student_habits_performance.csv')
students

In [ ]:
import random
col = ["ESP", "FRA", "DEU", "USA", "BRA", "CHN", "ZAF", "NGA", "IND", "KEN"]
new_col = []
for i in range(len(students)):
    a= random.randint(0,9)
    new_col.append(col[a])
students['country'] = pd.Series(new_col)
students

In [ ]:
df1=students.merge(gpd_pc_2021,on='country')
df1

In [ ]:
df1['exam_score_adjusted'] = df1['exam_score']*10000/df1['value_gpd']
df1